# Phase 4 (revised) — Hybrid Supervision: Per-Phase Optimal Strategy

We already have Phase 3 LSTM predictions for BOTH `all10` and `rank1` policies
across 5 years × 5 streams. We can reuse them to:

1. **Confirm the phase × supervision pattern is robust** across all 5 years and 5 streams.
2. **Construct a hybrid predictor**: for each phase, use whichever policy was best
   in cross-validation.

This costs zero new compute — it's all post-hoc analysis on existing predictions.

The Phase 3 LSTM results are in `phase3_outputs/lstm/lstm_predictions.npz`.


## 1. Setup

In [1]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from phase3_baselines.evaluate import evaluate_predictions

PREDS_NPZ = PROJECT_ROOT / 'phase3_outputs/lstm/lstm_predictions.npz'
RESULTS_CSV = PROJECT_ROOT / 'phase3_outputs/lstm/lstm_results.csv'
OUTPUT_DIR = PROJECT_ROOT / 'phase4_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2017, 2018, 2019, 2020, 2021]
STREAMS = ['S1_only', 'S2_only', 'Met_only', 'S1+S2', 'All']
PHASES = [10, 12, 15, 18, 19, 21, 24]


## 2. Load Phase 3 LSTM predictions and verify they are usable

In [2]:
print('Loading predictions npz...')
z = np.load(PREDS_NPZ, allow_pickle=False)
print(f'Number of arrays: {len(z.files)}')
# Print a sample of keys
sample_keys = sorted(z.files)[:5]
for k in sample_keys:
    print(f'  {k}: {z[k].shape}  {z[k].dtype}')


Loading predictions npz...
Number of arrays: 300
  All__all10__2017__bag: (5608,)  int64
  All__all10__2017__rank: (5608,)  int32
  All__all10__2017__xid: (5608,)  int64
  All__all10__2017__year: (5608,)  int32
  All__all10__2017__yp: (5608, 7)  int32


## 3. Cross-year robustness: per-phase MAE by (stream, policy)

In [3]:
# Use the existing results CSV which already has per-phase metrics
results = pd.read_csv(RESULTS_CSV)
print(f'Rows: {len(results)}')

# Mean MAE per (phase, stream, policy) across all 5 LOYO folds
mae_table = (results.groupby(['phase', 'stream', 'policy'])['mae']
                     .agg(['mean', 'std', 'count'])
                     .reset_index())
mae_pivot = mae_table.pivot_table(index=['phase', 'stream'], columns='policy',
                                    values='mean').round(2)
mae_pivot['Δ (all10 − rank1)'] = (mae_pivot['all10'] - mae_pivot['rank1']).round(2)
print('\n=== Per-phase MAE: all10 vs rank1 (averaged across LOYO folds) ===')
print(mae_pivot.to_string())


Rows: 350

=== Per-phase MAE: all10 vs rank1 (averaged across LOYO folds) ===
policy          all10  rank1  Δ (all10 − rank1)
phase stream                                   
10    All        9.76   9.91              -0.15
      Met_only  10.68  10.31               0.37
      S1+S2      9.44   9.57              -0.13
      S1_only   10.79  10.62               0.17
      S2_only    9.15  10.07              -0.92
12    All       11.04  10.31               0.73
      Met_only  11.56  10.65               0.91
      S1+S2     10.15  10.43              -0.28
      S1_only   11.04  12.75              -1.71
      S2_only   10.04  12.90              -2.86
15    All        9.33  10.24              -0.91
      Met_only   9.82  12.81              -2.99
      S1+S2      9.59  10.55              -0.96
      S1_only    9.73  11.29              -1.56
      S2_only    9.69  11.22              -1.53
18    All        5.31   7.45              -2.14
      Met_only   5.74   8.23              -2.49
      S1+S

In [4]:
# Identify which policy is best per phase for each stream
# Negative Δ = all10 wins; positive Δ = rank1 wins
print('\n=== Best policy per (phase, stream) ===')
flat = mae_table.pivot_table(index=['phase'], columns=['stream', 'policy'],
                              values='mean').round(2)
for phase in PHASES:
    print(f'\nPhase {phase}:')
    for stream in STREAMS:
        a = flat.loc[phase, (stream, 'all10')]
        r = flat.loc[phase, (stream, 'rank1')]
        winner = 'all10' if a < r else 'rank1'
        margin = abs(a - r)
        print(f'  {stream:9s}: all10={a:.2f}  rank1={r:.2f}  →  {winner}  (Δ={margin:.2f})')



=== Best policy per (phase, stream) ===

Phase 10:
  S1_only  : all10=10.79  rank1=10.62  →  rank1  (Δ=0.17)
  S2_only  : all10=9.15  rank1=10.07  →  all10  (Δ=0.92)
  Met_only : all10=10.68  rank1=10.31  →  rank1  (Δ=0.37)
  S1+S2    : all10=9.44  rank1=9.57  →  all10  (Δ=0.13)
  All      : all10=9.76  rank1=9.91  →  all10  (Δ=0.15)

Phase 12:
  S1_only  : all10=11.04  rank1=12.75  →  all10  (Δ=1.71)
  S2_only  : all10=10.04  rank1=12.90  →  all10  (Δ=2.86)
  Met_only : all10=11.56  rank1=10.65  →  rank1  (Δ=0.91)
  S1+S2    : all10=10.15  rank1=10.43  →  all10  (Δ=0.28)
  All      : all10=11.04  rank1=10.31  →  rank1  (Δ=0.73)

Phase 15:
  S1_only  : all10=9.73  rank1=11.29  →  all10  (Δ=1.56)
  S2_only  : all10=9.69  rank1=11.22  →  all10  (Δ=1.53)
  Met_only : all10=9.82  rank1=12.81  →  all10  (Δ=2.99)
  S1+S2    : all10=9.59  rank1=10.55  →  all10  (Δ=0.96)
  All      : all10=9.33  rank1=10.24  →  all10  (Δ=0.91)

Phase 18:
  S1_only  : all10=5.95  rank1=8.59  →  all10  (Δ=2.64)

## 4. Robustness: how often does each policy win per phase?

In [5]:
# For each (phase, year, stream), determine winner
fold_results = (results.groupby(['phase','stream','policy','test_year'])['mae']
                       .first()
                       .reset_index())
fold_pivot = fold_results.pivot_table(index=['phase','stream','test_year'],
                                        columns='policy', values='mae')
fold_pivot['winner'] = np.where(fold_pivot['all10'] < fold_pivot['rank1'],
                                 'all10', 'rank1')

# How often does all10 win per phase?
win_rate = (fold_pivot.reset_index()
                       .groupby('phase')['winner']
                       .apply(lambda s: (s == 'all10').mean())
                       .round(2))
print('=== Fraction of (year, stream) combinations where all10 wins ===')
print('(1.0 = all10 always wins; 0.0 = rank1 always wins)')
for p in PHASES:
    bar = '█' * int(win_rate.loc[p] * 30)
    print(f'  phase {p}: {win_rate.loc[p]:.2f}  {bar}')


=== Fraction of (year, stream) combinations where all10 wins ===
(1.0 = all10 always wins; 0.0 = rank1 always wins)
  phase 10: 0.68  ████████████████████
  phase 12: 0.68  ████████████████████
  phase 15: 0.92  ███████████████████████████
  phase 18: 0.92  ███████████████████████████
  phase 19: 0.92  ███████████████████████████
  phase 21: 0.88  ██████████████████████████
  phase 24: 0.80  ████████████████████████


In [6]:
# Same view but broken down by stream as well
win_by_stream = (fold_pivot.reset_index()
                            .groupby(['phase','stream'])['winner']
                            .apply(lambda s: (s == 'all10').mean())
                            .unstack('stream').round(2))
print('\n=== Fraction all10 wins, by (phase, stream) ===')
print(win_by_stream.to_string())



=== Fraction all10 wins, by (phase, stream) ===
stream  All  Met_only  S1+S2  S1_only  S2_only
phase                                         
10      0.6       0.6    0.6      0.6      1.0
12      0.4       0.6    0.6      0.8      1.0
15      0.8       1.0    0.8      1.0      1.0
18      1.0       0.8    1.0      0.8      1.0
19      1.0       0.8    1.0      0.8      1.0
21      1.0       0.8    0.8      0.8      1.0
24      0.8       0.6    0.8      0.8      1.0


## 5. Build the hybrid predictor

Strategy: for each phase, pick whichever policy had lower CV MAE on the **training folds**
(i.e., the LOYO folds where that test year was excluded). Then apply that policy's
prediction for the held-out year.

This is a proper hybrid (no test leakage): the policy choice per phase is made using
training-fold statistics only.


In [7]:
def build_hybrid(results_df, stream='All'):
    """For each (test_year, phase), choose the policy with lower mean MAE on the
    OTHER four years. Returns a DataFrame of hybrid per-phase MAE per year.
    """
    sub = results_df[results_df['stream'] == stream].copy()
    rows = []
    for test_y in YEARS:
        # 'training folds' = MAE from all OTHER test years
        for phase in PHASES:
            other = sub[(sub['test_year'] != test_y) & (sub['phase'] == phase)]
            mae_a10 = other[other['policy'] == 'all10']['mae'].mean()
            mae_r1  = other[other['policy'] == 'rank1']['mae'].mean()
            chosen_policy = 'all10' if mae_a10 < mae_r1 else 'rank1'
            # Now read the actual test MAE for the chosen policy
            test_mae = sub[(sub['test_year'] == test_y) &
                            (sub['phase'] == phase) &
                            (sub['policy'] == chosen_policy)]['mae'].values[0]
            rows.append({'test_year': test_y, 'phase': phase,
                          'chosen_policy': chosen_policy,
                          'mae': test_mae})
    return pd.DataFrame(rows)


# Build hybrid for each stream and compare
print('=== Hybrid predictor vs. all10 vs. rank1 (macro MAE across LOYO folds) ===')
comparison_rows = []
for stream in STREAMS:
    hyb = build_hybrid(results, stream=stream)
    hyb_mae = hyb['mae'].mean()
    # Reference numbers: pure all10 and pure rank1
    a10_mae = results[(results['stream'] == stream) &
                       (results['policy'] == 'all10')]['mae'].mean()
    r1_mae  = results[(results['stream'] == stream) &
                       (results['policy'] == 'rank1')]['mae'].mean()
    comparison_rows.append({'stream': stream,
                             'all10': round(a10_mae, 2),
                             'rank1': round(r1_mae, 2),
                             'hybrid': round(hyb_mae, 2),
                             'Δ vs all10': round(hyb_mae - a10_mae, 2),
                             'Δ vs rank1': round(hyb_mae - r1_mae, 2)})
comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False))
comparison.to_csv(OUTPUT_DIR / 'hybrid_macro_summary.csv', index=False)


=== Hybrid predictor vs. all10 vs. rank1 (macro MAE across LOYO folds) ===
  stream  all10  rank1  hybrid  Δ vs all10  Δ vs rank1
 S1_only   8.51   9.73    8.56        0.05       -1.18
 S2_only   8.03   9.78    8.03        0.00       -1.74
Met_only   9.10  10.73    9.35        0.25       -1.38
   S1+S2   7.88   8.73    7.91        0.02       -0.82
     All   8.07   8.81    8.21        0.14       -0.60


In [8]:
# Per-phase view of the hybrid choice (using stream=All)
print('\n=== Hybrid per-phase MAE (stream=All) vs reference policies ===')
hyb_all = build_hybrid(results, stream='All')
hyb_per_phase = hyb_all.groupby('phase')['mae'].mean().round(2)
a10_per_phase = results[(results['stream'] == 'All') &
                          (results['policy'] == 'all10')].groupby('phase')['mae'].mean().round(2)
r1_per_phase  = results[(results['stream'] == 'All') &
                          (results['policy'] == 'rank1')].groupby('phase')['mae'].mean().round(2)
phase_table = pd.DataFrame({'all10': a10_per_phase, 'rank1': r1_per_phase,
                             'hybrid': hyb_per_phase})
phase_table['min(a10,r1)'] = phase_table[['all10','rank1']].min(axis=1).round(2)
phase_table['hybrid optimal?'] = (phase_table['hybrid'] <= phase_table['min(a10,r1)'] + 0.01)
# Also which policy got picked
picked = hyb_all.groupby('phase')['chosen_policy'].agg(lambda s: s.value_counts().to_dict())
phase_table['picks'] = picked
print(phase_table.to_string())
phase_table.to_csv(OUTPUT_DIR / 'hybrid_per_phase_All.csv')



=== Hybrid per-phase MAE (stream=All) vs reference policies ===
       all10  rank1  hybrid  min(a10,r1)  hybrid optimal?                     picks
phase                                                                              
10      9.76   9.91   10.68         9.76            False  {'all10': 3, 'rank1': 2}
12     11.04  10.31   11.11        10.31            False  {'rank1': 4, 'all10': 1}
15      9.33  10.24    9.33         9.33             True              {'all10': 5}
18      5.31   7.45    5.31         5.31             True              {'all10': 5}
19      7.62   8.51    7.62         7.62             True              {'all10': 5}
21      7.04   8.05    7.04         7.04             True              {'all10': 5}
24      6.37   7.17    6.37         6.37             True              {'all10': 5}


## 6. Save the final aggregated results

In [9]:
# Save the full robustness table
robustness_summary = mae_pivot.reset_index()
robustness_summary.to_csv(OUTPUT_DIR / 'lstm_robustness_summary.csv', index=False)
print(f'Saved: {OUTPUT_DIR}/lstm_robustness_summary.csv')

# Save the per-phase winner table
fold_pivot.reset_index().to_csv(OUTPUT_DIR / 'lstm_fold_winners.csv', index=False)
print(f'Saved: {OUTPUT_DIR}/lstm_fold_winners.csv')

print('\n=== Done ===')
print('Key tables saved to', OUTPUT_DIR)


Saved: /fs1/home/atikul/Phenology/phase4_outputs/lstm_robustness_summary.csv
Saved: /fs1/home/atikul/Phenology/phase4_outputs/lstm_fold_winners.csv

=== Done ===
Key tables saved to /fs1/home/atikul/Phenology/phase4_outputs


## 7. Interpretation guide

What we want to see:

- **Section 4 (win rates)**: phases 10, 12 should have low all10-win-rate (rank1 wins most of the time).
  Phases 18, 19, 21, 24 should have high all10-win-rate (all10 wins most of the time).
  If the pattern is consistent across years and streams, the finding is robust.

- **Section 5 (hybrid)**: the hybrid predictor should beat the better of all10/rank1 on macro MAE
  for every stream. Modest expected gain: 0.3–0.7 days.

- **Section 5 per-phase**: the hybrid should produce close-to-optimal MAE for every phase
  (a few hundredths off due to the LOYO selection criterion not being perfect).
